In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.multioutput import MultiOutputClassifier

train_info = pd.read_csv("train_info.csv")
test_info = pd.read_csv("test_info.csv")
test_answer = pd.read_csv("test_answer.csv")

def chenge(x):
    return [int(i) for i in str(x).replace('[','').replace(']','').split() if i.strip() != '']

train_info['cut_point'] = train_info['cut_point'].apply(chenge)
test_info['cut_point'] = test_info['cut_point'].apply(chenge)

data_segments = []
data_segments2 = []


for idx, row in train_info.iterrows():
    unique_id = row['unique_id']
    cut_points = row['cut_point']
    mode = row['mode']
    file_path = f"train_data/{unique_id}.txt"

    sensor_df = pd.read_csv(file_path, header=None, sep=r'\s+', names=['Ax','Ay','Az','Gx','Gy','Gz'])
    sensor_df = sensor_df.apply(pd.to_numeric, errors='coerce').dropna()

    prev = 0
    for cp in cut_points:
        segment = sensor_df.iloc[prev:cp]
        prev = cp
        if len(segment) == 0:
            continue

        segment_mean = segment.mean()
        segment_std = segment.std()
        features = pd.concat([segment_mean, segment_std]).tolist()

        data_segments.append({
            'unique_id': unique_id,
            'mode': mode,
            'play years': row['play years'],
            'hold racket handed': row['hold racket handed'],
            'level': row['level'],
            'gender': row['gender'],
            **{f'feat{i}': features[i] for i in range(len(features))}
        })

for idx, row in test_info.iterrows():
    unique_id2 = row['unique_id']
    cut_points2 = row['cut_point']
    mode2 = row['mode']
    file_path2 = f"test_data/{unique_id2}.txt"

    sensor_df2 = pd.read_csv(file_path2, header=None, sep=r'\s+', names=['Ax','Ay','Az','Gx','Gy','Gz'])
    sensor_df2 = sensor_df2.apply(pd.to_numeric, errors='coerce').dropna()

    prev2 = 0
    for cp2 in cut_points2:
        segment2 = sensor_df2.iloc[prev2:cp2]
        prev2 = cp2
        if len(segment2) == 0:
            continue

        segment_mean2 = segment2.mean()
        segment_std2 = segment2.std()
        features2 = pd.concat([segment_mean2, segment_std2]).tolist()

        data_segments2.append({
            'unique_id': unique_id2,
            'mode': mode2,
            **{f'feat{i}': features2[i] for i in range(len(features2))}
        })


final_df = pd.DataFrame(data_segments)
final_df2 = pd.DataFrame(data_segments2)


X = final_df[[col for col in final_df.columns if col.startswith('feat')] + ['mode']].copy()
if X['mode'].dtype == 'object':
    X['mode'] = X['mode'].astype('category').cat.codes

y = final_df[['play years','hold racket handed','level','gender']].copy()
label_encoders = {}
for col in y.columns:
    if y[col].dtype == 'object':
        le = LabelEncoder()
        y[col] = le.fit_transform(y[col])
        label_encoders[col] = le

X2 = final_df2[[col for col in final_df2.columns if col.startswith('feat')] + ['mode']].copy()
if X2['mode'].dtype == 'object':
    X2['mode'] = X2['mode'].astype('category').cat.codes


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X2_scaled = scaler.transform(X2)


knn = KNeighborsClassifier(n_neighbors=5)
clf = MultiOutputClassifier(knn)
clf.fit(X_scaled, y)
y_pred3 = clf.predict(X2_scaled)


pred_df = final_df2[['unique_id']].copy()
for i, col in enumerate(y.columns):
    pred_df[col] = y_pred3[:, i]


agg_pred = pred_df.groupby('unique_id').agg(lambda x: x.mode()[0]).reset_index()


merged = pd.merge(test_answer, agg_pred, on='unique_id', suffixes=('_true', '_pred'))


for col in y.columns:
    print(f"{col} Accuracy:", accuracy_score(merged[f"{col}_true"], merged[f"{col}_pred"]))


play years Accuracy: 0.3958041958041958
hold racket handed Accuracy: 0.9965034965034965
level Accuracy: 0.4489510489510489
gender Accuracy: 0.7916083916083916
